# Day 45 — The gradient descent family: SGD, momentum, AdaGrad, RMSProp, Adam, AdamW

Day 44 ended on a training loop whose last line was `w -= lr * w.grad`. Everything in
this notebook replaces that one line, and nothing else changes:

```python
g = grad(w)
w = opt.step(w, g)
```

Two things make this more than a list of formulas. First, the learning rate has an
*exact* ceiling on a quadratic bowl - `lr < 2 / largest curvature` - and crossing it does
not raise, it just lets the loss run off to infinity. Second, momentum turns the step
count's dependence on the condition number from `kappa` into `sqrt(kappa)`, which is why
it is worth the extra line of state.

Then the details that are easy to get wrong and never report an error: Adam without bias
correction takes steps that are too *large* (not too small, as the folklore says),
`sqrt(v) + eps` is not `sqrt(v + eps)`, and L2 regularisation inside Adam decays the
weights that need it most the least.

In [1]:
import math

import numpy as np

np.seterr(all='ignore')        # a deliberately diverging run overflows

## 1. The update rules

Every optimiser below keeps whatever state it needs and exposes one method. Swapping
optimisers never touches the training loop.

In [2]:
# ================================================================ 1. the update rules
# Every optimiser has the same two-line interface, so the training loop never changes:
#     g = grad(w);  w = opt.step(w, g)
class SGD:
    """w -= lr * g, optionally with a velocity that remembers past gradients."""

    name = 'SGD'

    def __init__(self, lr=0.1, momentum=0.0, nesterov=False):
        self.lr, self.momentum, self.nesterov = lr, momentum, nesterov
        self.v = None
        if momentum:
            self.name = f'SGD+Nesterov({momentum})' if nesterov else f'SGD+momentum({momentum})'

    def step(self, w, g):
        if self.v is None:
            self.v = np.zeros_like(w)
        if not self.momentum:
            return w - self.lr * g
        self.v = self.momentum * self.v + g            # a running sum of gradients
        step = self.momentum * self.v + g if self.nesterov else self.v
        return w - self.lr * step


class AdaGrad:
    """Divide by the square root of every squared gradient ever seen."""

    name = 'AdaGrad'

    def __init__(self, lr=0.5, eps=1e-8):
        self.lr, self.eps, self.s = lr, eps, None

    def step(self, w, g):
        if self.s is None:
            self.s = np.zeros_like(w)
        self.s += g * g                                 # never decays: the step keeps shrinking
        return w - self.lr * g / (np.sqrt(self.s) + self.eps)


class RMSProp:
    """AdaGrad with a forgetting factor, so the step size can recover."""

    name = 'RMSProp'

    def __init__(self, lr=0.05, rho=0.9, eps=1e-8):
        self.lr, self.rho, self.eps, self.s = lr, rho, eps, None

    def step(self, w, g):
        if self.s is None:
            self.s = np.zeros_like(w)
        self.s = self.rho * self.s + (1 - self.rho) * g * g
        return w - self.lr * g / (np.sqrt(self.s) + self.eps)


class Adam:
    """Momentum on the gradient (m) and on its square (v), each bias-corrected.

    The flags exist to show what breaks when a detail is dropped:
      bias_correction=False  the first steps are far too small
      eps_inside=True        sqrt(v + eps) instead of sqrt(v) + eps
      weight_decay, decoupled   L2-in-the-gradient (Adam) vs AdamW
    """

    def __init__(self, lr=0.05, b1=0.9, b2=0.999, eps=1e-8, bias_correction=True,
                 eps_inside=False, weight_decay=0.0, decoupled=False):
        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps
        self.bias_correction, self.eps_inside = bias_correction, eps_inside
        self.weight_decay, self.decoupled = weight_decay, decoupled
        self.m = self.v = None
        self.t = 0
        self.name = 'AdamW' if (weight_decay and decoupled) else 'Adam'
        if weight_decay and not decoupled:
            self.name = 'Adam + L2'

    def step(self, w, g):
        if self.m is None:
            self.m = np.zeros_like(w)
            self.v = np.zeros_like(w)
        self.t += 1
        if self.weight_decay and not self.decoupled:
            g = g + self.weight_decay * w              # L2 penalty folded into the gradient
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g * g
        if self.bias_correction:
            mh = self.m / (1 - self.b1 ** self.t)
            vh = self.v / (1 - self.b2 ** self.t)
        else:
            mh, vh = self.m, self.v
        denom = np.sqrt(vh + self.eps) if self.eps_inside else np.sqrt(vh) + self.eps
        w = w - self.lr * mh / denom
        if self.weight_decay and self.decoupled:
            w = w - self.lr * self.weight_decay * w    # decay applied straight to w
        return w


print([o.name for o in (SGD(), SGD(0.1, 0.9), SGD(0.1, 0.9, True), AdaGrad(), RMSProp(),
                        Adam(), Adam(weight_decay=0.1), Adam(weight_decay=0.1, decoupled=True))])

['SGD', 'SGD+momentum(0.9)', 'SGD+Nesterov(0.9)', 'AdaGrad', 'RMSProp', 'Adam', 'Adam + L2', 'AdamW']


## 2. A bowl with a known answer

`f(w) = 0.5 * (w0² + 20 w1²)` is 20 times more curved across than along. Everything about
plain gradient descent on it is known in closed form, so the measurements below can be
checked rather than believed.

In [3]:
# ================================================================ 2. the test landscape
class Quadratic:
    """f(w) = 0.5 * sum(curv * w^2): a bowl stretched into a ravine by `curv`.

    Everything about plain gradient descent on this bowl is known in closed form,
    so the measured step counts can be checked against the theory.
    """

    def __init__(self, curv=(1.0, 20.0)):
        self.curv = np.asarray(curv, dtype=float)

    def loss(self, w):
        return float(0.5 * np.sum(self.curv * w * w))

    def grad(self, w):
        return self.curv * w

    @property
    def lmax(self):
        return float(self.curv.max())

    @property
    def lmin(self):
        return float(self.curv.min())

    @property
    def kappa(self):
        """Condition number: how much longer the ravine is than it is wide."""
        return self.lmax / self.lmin

    @property
    def lr_divergence(self):
        """Plain GD on a quadratic converges exactly while lr < 2 / lmax."""
        return 2.0 / self.lmax

    @property
    def lr_best(self):
        return 2.0 / (self.lmin + self.lmax)

    @property
    def rate_gd(self):
        """Error multiplier per step at the best lr."""
        return (self.kappa - 1) / (self.kappa + 1)

    @property
    def rate_momentum(self):
        """Heavy-ball with the best beta: the same expression with sqrt(kappa)."""
        r = math.sqrt(self.kappa)
        return (r - 1) / (r + 1)

    @property
    def beta_best(self):
        return self.rate_momentum ** 2

    @property
    def lr_momentum_best(self):
        """Heavy-ball wants its own step size, not gradient descent's."""
        return (2.0 / (math.sqrt(self.lmax) + math.sqrt(self.lmin))) ** 2

    def steps_for(self, rate, tol=1e-6):
        """How many steps a per-step error multiplier `rate` needs to reach tol."""
        return math.log(tol) / math.log(rate)


def run(opt, prob, w0, steps=200):
    """Return the trajectory of w, one row per step (including the start)."""
    w = np.asarray(w0, dtype=float).copy()
    out = [w.copy()]
    for _ in range(steps):
        w = opt.step(w, prob.grad(w))
        if not np.all(np.isfinite(w)):
            out.append(w.copy())
            break
        out.append(w.copy())
    return np.array(out)


def steps_to(opt, prob, w0, tol=1e-6, cap=100_000):
    """How many steps until |w| falls below tol.  cap+1 means 'never got there'."""
    w = np.asarray(w0, dtype=float).copy()
    start = float(np.linalg.norm(w))
    for k in range(1, cap + 1):
        w = opt.step(w, prob.grad(w))
        if not np.all(np.isfinite(w)):
            return cap + 1
        if np.linalg.norm(w) <= tol * start:
            return k
    return cap + 1


def multiplier(lr, prob):
    """Plain GD on a quadratic multiplies each coordinate by (1 - lr * curvature)
    every step.  The whole run converges exactly when all of those are below 1."""
    return float(np.max(np.abs(1.0 - lr * prob.curv)))


def diverges(lr, prob, w0, steps=20_000):
    """Run it and look, rather than trusting the formula."""
    w = np.asarray(w0, dtype=float).copy()
    start = float(np.linalg.norm(w))
    for _ in range(steps):
        w = w - lr * prob.grad(w)
        if not np.all(np.isfinite(w)) or np.linalg.norm(w) > 1e3 * start:
            return True
    return False


q = Quadratic()
w0 = np.array([1.0, 1.0])
print(f'condition number {q.kappa:g}, lr ceiling 2/{q.lmax:g} = {q.lr_divergence:g}')
for lr in (0.05, 0.0999, 0.1001, 0.11):
    print(f'  lr {lr:<7g} multiplier {multiplier(lr, q):.5f}  '
          f'{"diverges" if diverges(lr, q, w0) else "converges"}')
print('after 400 steps at lr 0.11:', run(SGD(lr=0.11), q, w0, 400)[-1])

condition number 20, lr ceiling 2/20 = 0.1
  lr 0.05    multiplier 0.95000  converges
  lr 0.0999  multiplier 0.99800  converges
  lr 0.1001  multiplier 1.00200  diverges
  lr 0.11    multiplier 1.20000  diverges
after 400 steps at lr 0.11: [5.70167762e-21 4.70433693e+31]


## 3. Momentum: kappa becomes sqrt(kappa)

Plain gradient descent at its best learning rate shrinks the error by
`(kappa - 1)/(kappa + 1)` per step. Heavy-ball momentum, with its own best learning rate
and beta, gets `(sqrt(kappa) - 1)/(sqrt(kappa) + 1)`. That is the entire reason momentum
exists.

In [4]:
print(f'plain GD   lr {q.lr_best:.4f}  rate {q.rate_gd:.4f}  '
      f'theory {q.steps_for(q.rate_gd):.1f} steps  measured {steps_to(SGD(lr=q.lr_best), q, w0)}')
mo = SGD(lr=q.lr_momentum_best, momentum=q.beta_best)
print(f'momentum   lr {q.lr_momentum_best:.4f}  beta {q.beta_best:.4f}  rate {q.rate_momentum:.4f}  '
      f'theory {q.steps_for(q.rate_momentum):.1f} steps  measured {steps_to(mo, q, w0)}')
print('beta is not a free lunch: copying beta=0.9 from a blog post, on this problem,')
for beta in (0.0, 0.4026, 0.9):
    print(f'  lr 0.04, beta {beta:<6g} -> {steps_to(SGD(lr=0.04, momentum=beta), q, [4.0, 1.0])} steps')

plain GD   lr 0.0952  rate 0.9048  theory 138.0 steps  measured 139
momentum   lr 0.1336  beta 0.4026  rate 0.6345  theory 30.4 steps  measured 39
beta is not a free lunch: copying beta=0.9 from a blog post, on this problem,
  lr 0.04, beta 0      -> 338 steps
  lr 0.04, beta 0.4026 -> 190 steps
  lr 0.04, beta 0.9    -> 245 steps


## 4. A real problem, with each learning rate tuned

Comparing optimisers at one shared learning rate is not a fair fight: AdaGrad, RMSProp
and Adam divide the gradient by its own size, so their `lr` means a different thing.
Each one gets its own sweep here.

In [5]:
# ================================================================ 3. a real problem, in mini-batches
class LeastSquares:
    """(1/n) * ||Xw - y||^2.  One column is deliberately scaled up, so the bowl is a
    ravine for exactly the same reason the toy quadratic is."""

    def __init__(self, n=400, scales=(1.0, 1.0, 8.0), noise=0.3, seed=45):
        rng = np.random.default_rng(seed)
        d = len(scales)
        self.X = rng.normal(size=(n, d)) * np.asarray(scales)
        self.w_true = np.array([2.0, -1.0, 0.5])[:d]
        self.y = self.X @ self.w_true + noise * rng.normal(size=n)
        self.n = n
        self.w_star = np.linalg.lstsq(self.X, self.y, rcond=None)[0]   # the exact optimum
        self.best = float(np.mean((self.X @ self.w_star - self.y) ** 2))

    def grad(self, w, idx=None):
        X = self.X if idx is None else self.X[idx]
        y = self.y if idx is None else self.y[idx]
        return 2.0 / len(y) * (X.T @ (X @ w - y))

    def loss(self, w):
        """Excess loss over the exact least-squares optimum, so it goes to zero."""
        return float(np.mean((self.X @ w - self.y) ** 2)) - self.best

    @property
    def kappa(self):
        c = self.curvature
        return float(c.max() / c.min())

    @property
    def curvature(self):
        return np.linalg.eigvalsh(2.0 / self.n * self.X.T @ self.X)

    @property
    def lr_divergence(self):
        return 2.0 / float(self.curvature.max())

    def sorted_by_target(self):
        """The same data with the rows in sorted order - the shape real data arrives in
        when it was grouped by label or collected over time."""
        import copy
        other = copy.deepcopy(self)
        idx = np.argsort(other.y)
        other.X, other.y = other.X[idx], other.y[idx]
        return other

    @property
    def lr_divergence_single(self):
        """A single row has its own, much sharper curvature: 2 * ||x||^2.  A batch of
        one has to respect that, not the average - which is why the smaller the batch,
        the smaller the learning rate it survives."""
        worst = float(np.max(2.0 * np.sum(self.X * self.X, axis=1)))
        return 2.0 / worst


def steps_to_loss(opt, prob, w0, tol=1e-8, cap=5000):
    """Full-batch steps until the excess loss drops below tol."""
    w = np.asarray(w0, dtype=float).copy()
    for k in range(1, cap + 1):
        with np.errstate(over='ignore', invalid='ignore'):
            w = opt.step(w, prob.grad(w))
        if not np.all(np.isfinite(w)):
            return cap + 1
        if prob.loss(w) <= tol:
            return k
    return cap + 1


def best_lr_loss(make_opt, prob, w0, grid=None, tol=1e-8, cap=3000):
    if grid is None:
        grid = [10 ** e for e in np.arange(-3.0, 1.51, 0.1)]
    best = (None, cap + 1)
    for lr in grid:
        k = steps_to_loss(make_opt(lr), prob, w0, tol=tol, cap=cap)
        if k < best[1]:
            best = (lr, k)
    return best


FAMILIES = [
    ('SGD', lambda lr: SGD(lr)),
    ('SGD + momentum 0.9', lambda lr: SGD(lr, 0.9)),
    ('SGD + Nesterov 0.9', lambda lr: SGD(lr, 0.9, True)),
    ('AdaGrad', lambda lr: AdaGrad(lr)),
    ('RMSProp', lambda lr: RMSProp(lr)),
    ('Adam', lambda lr: Adam(lr)),
]


ls = LeastSquares()
z = np.zeros(3)
print(f'condition number {ls.kappa:.1f}, lr ceiling {ls.lr_divergence:.4f}')
for name, make in FAMILIES:
    lr, k = best_lr_loss(make, ls, z)
    print(f'  {name:<20} best lr {lr:>7.4f}   {k:>4} steps')

condition number 77.1, lr ceiling 0.0140
  SGD                  best lr  0.0126    400 steps
  SGD + momentum 0.9   best lr  0.0032    181 steps
  SGD + Nesterov 0.9   best lr  0.0079    163 steps
  AdaGrad              best lr  1.5849     10 steps
  RMSProp              best lr  0.1585     26 steps
  Adam                 best lr  0.6310    153 steps


## 5. Adam without bias correction steps too far

`m` and `v` both start at zero, so both are too small early. They do not cancel, because
`v` sits under a square root: the uncorrected step is `(1 - 0.9^t) / sqrt(1 - 0.999^t)`
times the corrected one, which is **above** 1 for thousands of steps.

In [6]:
# ================================================================ 6. Adam's details
def bias_factor(t, b1=0.9, b2=0.999):
    """How much larger an un-corrected Adam step is than a corrected one, at step t.

    corrected  = (m / (1 - b1^t)) / sqrt(v / (1 - b2^t))
    uncorrected = m / sqrt(v)
    so the ratio is (1 - b1^t) / sqrt(1 - b2^t), which is *above* 1 for a long time.
    """
    return (1 - b1 ** t) / math.sqrt(1 - b2 ** t)


def first_steps(opt, prob, w0, n=3):
    """The size of the first n parameter updates."""
    w = np.asarray(w0, dtype=float).copy()
    sizes = []
    for _ in range(n):
        nw = opt.step(w, prob.grad(w))
        sizes.append(float(np.linalg.norm(nw - w)))
        w = nw
    return sizes


for t in (1, 12, 100, 1000, 5000):
    print(f'  step {t:>5}: {bias_factor(t):.3f}x')
print('measured first three steps at lr 0.05:')
print('  with correction   ', [round(x, 4) for x in first_steps(Adam(lr=0.05), ls, z)])
print('  without           ', [round(x, 4) for x in first_steps(Adam(lr=0.05, bias_correction=False), ls, z)])

  step     1: 3.162x
  step    12: 6.569x
  step   100: 3.241x
  step  1000: 1.258x
  step  5000: 1.003x
measured first three steps at lr 0.05:
  with correction    [0.0866, 0.0864, 0.0861]
  without            [0.2739, 0.3631, 0.4038]


## 6. Where the epsilon goes, and why Adam + L2 is not AdamW

`sqrt(v) + eps` and `sqrt(v + eps)` look interchangeable. With `eps = 1e-8`,
`sqrt(eps) = 1e-4`, so on coordinates with tiny gradients the second form shrinks the
step by orders of magnitude. PyTorch uses the first.

L2 regularisation adds `wd * w` to the gradient, and Adam then divides the whole gradient
by `sqrt(v)`. So the decay applied to each weight is scaled by `1 / sqrt(v)`: the weights
with the largest gradients are decayed the least. AdamW subtracts `lr * wd * w` from `w`
directly instead.

In [7]:
def final_weights(opt, prob, w0, steps=400):
    w = np.asarray(w0, dtype=float).copy()
    for _ in range(steps):
        w = opt.step(w, prob.grad(w))
    return w


for v in (1e-4, 1e-8, 1e-12, 1e-16):
    a, b = math.sqrt(v) + 1e-8, math.sqrt(v + 1e-8)
    print(f'  v = {v:<8g} sqrt(v)+eps {a:.3e}   sqrt(v+eps) {b:.3e}   ratio {b / a:8.1f}x')

opt = Adam(lr=0.05)
w = z.copy()
for _ in range(400):
    w = opt.step(w, ls.grad(w))
rv = np.sqrt(opt.v)
rel = (1 / rv) / np.mean(1 / rv)
print('sqrt(v) per weight:', np.round(rv, 3))
print('Adam + L2 decay strength (average 1):', np.round(rel, 3), ' AdamW: [1. 1. 1.]')
print(f'the big-gradient weight is decayed {rel[0] / rel[2]:.0f}x less than weight 0')

  v = 0.0001   sqrt(v)+eps 1.000e-02   sqrt(v+eps) 1.000e-02   ratio      1.0x
  v = 1e-08    sqrt(v)+eps 1.000e-04   sqrt(v+eps) 1.414e-04   ratio      1.4x
  v = 1e-12    sqrt(v)+eps 1.010e-06   sqrt(v+eps) 1.000e-04   ratio     99.0x
  v = 1e-16    sqrt(v)+eps 2.000e-08   sqrt(v+eps) 1.000e-04   ratio   5000.0x
sqrt(v) per weight: [0.407 0.146 4.059]
Adam + L2 decay strength (average 1): [0.771 2.152 0.077]  AdamW: [1. 1. 1.]
the big-gradient weight is decayed 10x less than weight 0


## 7. Mini-batches: the lr ceiling moves, and the loss stops improving

A batch of one has to respect the curvature of its own row, which is far sharper than the
average over all rows. Below that ceiling a small batch covers much more ground per pass
over the data - and then stops, circling the optimum at a radius the step size sets.

In [8]:
def sgd_epochs(opt, prob, w0, batch, epochs, shuffle=True, seed=0):
    """Mini-batch training.  Returns the loss after every epoch."""
    rng = np.random.default_rng(seed)
    w = np.asarray(w0, dtype=float).copy()
    order = np.arange(prob.n)
    hist = []
    for _ in range(epochs):
        if shuffle:
            rng.shuffle(order)
        for s in range(0, prob.n, batch):
            idx = order[s:s + batch]
            w = opt.step(w, prob.grad(w, idx))
            if not np.all(np.isfinite(w)):
                hist.append(float('inf'))
                return w, hist
        hist.append(prob.loss(w))
    return w, hist


print(f'ceiling from the average {ls.lr_divergence:.4f}, from the sharpest single row '
      f'{ls.lr_divergence_single:.5f} ({ls.lr_divergence / ls.lr_divergence_single:.1f}x smaller)')
for b in (1, 8, 32, ls.n):
    h = sgd_epochs(SGD(lr=0.01), ls, z, batch=b, epochs=30)[1][-1]
    print(f'  lr 0.01, batch {b:<4} after 30 epochs: {"blew up" if h > 1e6 else f"{h:.1e}"}')
for ep in (10, 30, 100, 300):
    h8 = sgd_epochs(SGD(lr=0.002), ls, z, batch=8, epochs=ep)[1][-1]
    hf = sgd_epochs(SGD(lr=0.002), ls, z, batch=ls.n, epochs=ep)[1][-1]
    print(f'  after {ep:>3} epochs   batch 8 {h8:.3e}   full batch {hf:.3e}')
srt = ls.sorted_by_target()
a = np.median([sgd_epochs(SGD(lr=0.002), srt, z, 8, 30, True, s)[1][-1] for s in range(5)])
b = np.median([sgd_epochs(SGD(lr=0.002), srt, z, 8, 30, False, s)[1][-1] for s in range(5)])
print(f'  rows sorted by target: reshuffled {a:.3e}, fixed order {b:.3e} ({b / a:.1f}x)')

ceiling from the average 0.0140, from the sharpest single row 0.00165 (8.5x smaller)
  lr 0.01, batch 1    after 30 epochs: blew up
  lr 0.01, batch 8    after 30 epochs: 4.6e-04
  lr 0.01, batch 32   after 30 epochs: 4.8e-03
  lr 0.01, batch 400  after 30 epochs: 1.5e+00
  after  10 epochs   batch 8 8.813e-02   full batch 4.626e+00
  after  30 epochs   batch 8 2.545e-03   full batch 3.911e+00
  after 100 epochs   batch 8 5.396e-03   full batch 2.210e+00
  after 300 epochs   batch 8 2.484e-03   full batch 4.352e-01
  rows sorted by target: reshuffled 2.073e-04, fixed order 2.516e-03 (12.1x)


## 8. LeetCode 462 and LeetCode 852

Two problems that are optimisation without a gradient. 462 minimises
`sum|x - c|`, whose minimiser is the median, not the mean. 852 finds the top of a mountain
by binary-searching the *sign of the slope* - gradient ascent that halves the search space
instead of taking a step.

In [9]:
# ================================================================ 4. LeetCode 462
class Solution462:
    """Minimise the total distance to a single point: the answer is the median."""

    def minMoves2(self, nums: list[int]) -> int:
        nums = sorted(nums)
        mid = nums[len(nums) // 2]
        return sum(abs(x - mid) for x in nums)


def moves_to(nums, c):
    return sum(abs(x - c) for x in nums)


def minmoves_mean(nums):
    """The tempting wrong answer: the mean minimises squared error, not absolute error."""
    c = round(sum(nums) / len(nums))
    return moves_to(nums, c)


def subgradient_median(nums, lr=0.5, steps=400, c0=0.0):
    """Gradient descent on sum|x - c|.  The derivative of |x - c| is the sign, which is
    why this walks to the median rather than the mean - and why it never settles
    exactly: the step size, not the slope, decides where it stops."""
    c = float(c0)
    traj = [c]
    for k in range(1, steps + 1):
        g = float(sum(np.sign(c - x) for x in nums))
        c -= lr / math.sqrt(k) * g            # a 1/sqrt(k) schedule, or it oscillates forever
        traj.append(c)
    return c, traj


# ================================================================ 5. LeetCode 852
class Solution852:
    """Find the top of a mountain by binary-searching the sign of the slope."""

    def peakIndexInMountainArray(self, arr: list[int]) -> int:
        lo, hi = 0, len(arr) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if arr[mid] < arr[mid + 1]:       # still going up: the peak is to the right
                lo = mid + 1
            else:                             # already coming down: mid could be the peak
                hi = mid
        return lo


def peak_linear(arr):
    """O(n): the answer a scan gives, used only to check the binary search."""
    return int(np.argmax(arr))


def peak_probes(arr):
    """How many elements the binary search actually looks at."""
    lo, hi, probes = 0, len(arr) - 1, 0
    while lo < hi:
        mid = (lo + hi) // 2
        probes += 2
        if arr[mid] < arr[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return probes


def ternary_min(f, lo, hi, iters=100):
    """The continuous version: no derivative, just 'which third can I throw away'."""
    for _ in range(iters):
        a = lo + (hi - lo) / 3
        b = hi - (hi - lo) / 3
        if f(a) < f(b):
            hi = b
        else:
            lo = a
    return (lo + hi) / 2


def mountain(n=4001, peak=None, seed=45):
    """A strictly-increasing-then-decreasing array."""
    peak = n // 3 if peak is None else peak
    up = np.arange(1, peak + 2) * 3
    down = up[-1] - np.arange(1, n - peak) * 2
    arr = np.concatenate([up, down]).astype(int)
    return arr.tolist(), peak


for nums in ([1, 2, 3], [1, 10, 2, 9], [1, 2, 3, 100]):
    print(f'  {str(nums):<16} median {Solution462().minMoves2(nums):>4}   '
          f'mean instead {minmoves_mean(nums):>4}')
c, _ = subgradient_median([1, 2, 3, 100])
print(f'  gradient descent on sum|x - c| lands at {c:.3f}, cost {moves_to([1,2,3,100], c):.1f}')
arr, peak = mountain()
print(f'  mountain of {len(arr)}: peak {Solution852().peakIndexInMountainArray(arr)} '
      f'(scan says {peak_linear(arr)}), {peak_probes(arr)} elements looked at')
print(f'  ternary search on a continuous unimodal f: '
      f'{ternary_min(lambda x: (x - 1.234567) ** 2 + 3, -10, 10):.6f}')

  [1, 2, 3]        median    2   mean instead    2
  [1, 10, 2, 9]    median   16   mean instead   16
  [1, 2, 3, 100]   median  100   mean instead  146
  gradient descent on sum|x - c| lands at 2.354, cost 100.0
  mountain of 4001: peak 1333 (scan says 1333), 24 elements looked at
  ternary search on a continuous unimodal f: 1.234567


## Tests

In [10]:
def check() -> None:
    np.seterr(all='ignore')
    q = Quadratic()
    w0 = np.array([1.0, 1.0])
    assert q.kappa == 20.0 and q.lr_divergence == 0.1
    assert abs(q.lr_best - 2 / 21) < 1e-12
    # the ceiling is exactly 2 / lmax, on both sides
    assert not diverges(0.0999, q, w0) and diverges(0.1001, q, w0)
    assert multiplier(0.0999, q) < 1.0 < multiplier(0.1001, q)
    # theory matches measurement to within a step or two
    assert abs(steps_to(SGD(lr=q.lr_best), q, w0) - q.steps_for(q.rate_gd)) <= 2
    m_meas = steps_to(SGD(lr=q.lr_momentum_best, momentum=q.beta_best), q, w0)
    assert m_meas < steps_to(SGD(lr=q.lr_best), q, w0) / 3
    # un-corrected Adam steps are larger, not smaller, and stay wrong for a long time
    assert bias_factor(1) > 3 and max(bias_factor(t) for t in range(1, 2001)) > 6
    assert bias_factor(5000) < 1.01
    ls = LeastSquares()
    z = np.zeros(3)
    assert ls.loss(ls.w_star) <= 1e-12 and ls.loss(z) > 0
    assert first_steps(Adam(lr=0.05, bias_correction=False), ls, z)[0] > \
           first_steps(Adam(lr=0.05), ls, z)[0]
    # eps inside the sqrt makes a far bigger denominator when v is small
    assert math.sqrt(1e-12 + 1e-8) / (math.sqrt(1e-12) + 1e-8) > 50
    # Adam + L2 decays the big-gradient weight least
    w_l2 = final_weights(Adam(lr=0.05, weight_decay=0.1), ls, z)
    w_wd = final_weights(Adam(lr=0.05, weight_decay=0.1, decoupled=True), ls, z)
    w_no = final_weights(Adam(lr=0.05), ls, z)
    assert np.linalg.norm(w_wd) < np.linalg.norm(w_no)
    assert abs(w_l2[2]) > abs(w_wd[2])
    # a single row is far sharper than the average, so batch 1 needs a smaller lr
    assert ls.lr_divergence_single < ls.lr_divergence / 5
    _, h1 = sgd_epochs(SGD(lr=0.01), ls, z, batch=1, epochs=30)
    _, hf = sgd_epochs(SGD(lr=0.01), ls, z, batch=ls.n, epochs=30)
    assert h1[-1] > 1e6 and math.isfinite(hf[-1])
    # below that ceiling a small batch gets further per pass over the data
    _, h8 = sgd_epochs(SGD(lr=0.002), ls, z, batch=8, epochs=30)
    _, hfull = sgd_epochs(SGD(lr=0.002), ls, z, batch=ls.n, epochs=30)
    assert h8[-1] < hfull[-1] / 100
    # ... and then plateaus while full batch keeps going
    _, h8_long = sgd_epochs(SGD(lr=0.002), ls, z, batch=8, epochs=300)
    assert h8_long[-1] > h8[-1] / 10
    # shuffling only matters once the rows arrive in a meaningful order
    srt = ls.sorted_by_target()
    sh = np.median([sgd_epochs(SGD(lr=0.002), srt, z, 8, 30, True, sd)[1][-1] for sd in range(5)])
    fx = np.median([sgd_epochs(SGD(lr=0.002), srt, z, 8, 30, False, sd)[1][-1] for sd in range(5)])
    assert fx > 5 * sh
    # every tuned optimiser actually reaches the tolerance
    for _, make in FAMILIES:
        assert best_lr_loss(make, ls, z)[1] <= 3000
    # LeetCode
    s462 = Solution462()
    assert s462.minMoves2([1, 2, 3]) == 2 and s462.minMoves2([1, 10, 2, 9]) == 16
    assert s462.minMoves2([1, 2, 3, 100]) == 100 and minmoves_mean([1, 2, 3, 100]) == 146
    for nums in ([1, 2, 3], [1, 10, 2, 9], [1, 2, 3, 100], [5]):
        best = min(moves_to(nums, c) for c in range(min(nums), max(nums) + 1))
        assert s462.minMoves2(nums) == best
    c, _ = subgradient_median([1, 2, 3, 100])
    assert 2.0 <= c <= 3.0
    s852 = Solution852()
    for n, pk in ((5, 2), (101, 50), (4001, 1333), (3, 1)):
        arr, p = mountain(n, pk)
        assert s852.peakIndexInMountainArray(arr) == p == peak_linear(arr)
    arr, _ = mountain(4001)
    assert peak_probes(arr) <= 2 * math.ceil(math.log2(len(arr))) + 2
    assert abs(ternary_min(lambda x: (x - 1.234567) ** 2 + 3, -10, 10) - 1.234567) < 1e-6


check()
print('all assertions passed')

all assertions passed
